# 第3回 欠測値への対応や標準化など前処理の実行（解答例・教員用）


---


## 前回からの接続


第2回は、自転車シェアの台数を1本の式で当て、データを訓練とテストに分けて、まだ見ていないデータで実力を測りました。ただ、自転車のデータは、欠けた値もなく、どの列も数の、整えられたデータでした。

今日から題材が変わります。現場のデータには、極端な値が混ざり、値が欠け、文字の列が並び、列ごとに目盛りがばらばらです。そのままでは式に入れられないデータを、式に入れられる形に整えます。この作業を前処理（preprocessing）と呼びます。


---


## 今日の分析目標


あなたは民泊の新しいホストです。自分のマンションの一室を貸切で出すとき、1泊いくらの値段を付ければいいかに答える。

東京のAirbnbに掲載された約3万2千件の部屋のデータを整えて、料金を当てる線形回帰（linear regression）の式を作り、あなたの部屋に値段を付けます。今日いちばん時間をかけるのは、欠けた値の扱いです。欠けた理由を確かめ、補い方の良し悪しを、答えの分かる欠けで採点します。

もう1つのねらいは、前処理を関数（`def`）にまとめ、引数を変えて使いこなせるようになることです。問題を1つ見つけるたびに小さな関数を1つ書き、最後にそれを組み合わせて、訓練・テスト・あなたの部屋に同じ整え方を当てます。最後の「目標に答えられたか」で振り返りましょう。


### 場面

あなたは、民泊を始めたばかりの新しいホストです。東京の自分のマンションの一室を、貸切で出したいと考えています。定員や寝室の数は、もう決まっています。

決めたいのは、掲載するときの1泊の値段です。その値段がどれくらい外れうるかも、知っておきたいところです。


### 手元のデータ

Inside Airbnbが公開している、東京のAirbnbの掲載32,361件です（2026年6月30日時点）。1行が1つの部屋で、地区・部屋のタイプ・定員・寝室数・ベッド数などが並びます。答えの列は、その部屋の1泊の料金（円）です。

第2回のデータとは違い、1泊500万円を超える部屋が混ざり、寝室数やバスルーム数がところどころ欠けています。バスルーム数を「1.5 baths」のような文字で書いた列もあります。


### 最後に出す答え

あなたの部屋に付ける1泊の値段（円）と、その値段が、まだ見ていない部屋で平均何円外れるかです。


---


## 学習ゴール


この回を終えると、次のことができるようになります。

- 極端な値や、目標と別の種類の行を外すかどうかを、分布を見て判断し、その理由を言葉で説明できる
- 何度も使う処理を`def`で関数にまとめ、引数（既定値つきを含む）と戻り値を読み書きできる
- 欠測（missing value）の量と欠けた理由を確かめ、別の列から戻す・埋めずに使わない・ほかの列から推測して補う、を選べる
- 平均で埋める補完（imputation）がなぜよくないかを、答えの分かる欠けで採点して説明できる
- 標準化（standardization）で列の目盛りをそろえ、線形回帰の予測は変わらないことを確かめられる
- 補い方を訓練データだけで決め、関数にして、テストと新しい部屋に同じように当てられる


---


## 0. 準備


### 最初に実行する：道具を呼び出す

- 役割：このノートで使う道具（ライブラリ）を呼び出す
- 入力：なし
- 出力：なし（終わるとセルの左に ✓ が付く）

`import`は道具箱を持ってくる命令、`from`は道具箱から1つだけ取り出す書き方です。今日も、numpy（`np`）・pandas（`pd`）・matplotlib（`plt`）・scikit-learnの4つを使います。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split


### 図の日本語を表示できるようにする

- 役割：図の中の日本語が □ にならないようにし、図の大きさと字の大きさをそろえる
- 入力：なし
- 出力：「セットアップ完了」の表示

`try:`から先は、日本語表示の道具が動かなくなったときの保険で、中身は読まなくてかまいません。


In [ ]:
# 日本語が図の中で □ になるのを防ぐ
!pip install -q japanize-matplotlib
try:
    import japanize_matplotlib
# japanize が動かなくなったときの保険：同梱フォントを直接登録する
except Exception:
    from importlib.util import find_spec
    from matplotlib import font_manager as fm
    from pathlib import Path
    spec = find_spec('japanize_matplotlib')
    ttf = None
    if spec:
        ttf = next(Path(spec.origin).parent.rglob('*.ttf'), None)
    if ttf:
        fm.fontManager.addfont(str(ttf))
        name = fm.FontProperties(fname=str(ttf)).get_name()
        plt.rcParams['font.family'] = name

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False  # マイナス記号の化けを防ぐ
print('セットアップ完了')


### お手本：表を読み込んで、列名を日本語にする

- 役割：CSVファイルの表を読み込み、英語の列名を日本語に付け替えて、先頭5行を見る
- 入力：データの置き場所`DATA_DIR`と、ファイル名`airbnb_tokyo.csv`
- 出力：表`raw`（このあと全部の節で使う元の表）

`{'元の名前': '新しい名前', ...}`の辞書を`rename(columns=...)`に渡すと、その対応どおりに列名を付け替えます。値は変わりません。元の表は、あとで何度も使うので`raw`（生のデータ）という名前にしておきます。


In [ ]:
# データはこのリポジトリから読み込む
DATA_DIR = ('https://raw.githubusercontent.com/k0heiun0/'
            'applied_exercise_preview/main/shared/data')
raw = pd.read_csv(f'{DATA_DIR}/airbnb_tokyo.csv')
ja_names = {'neighbourhood': '地区', 'room_type': '部屋のタイプ',
            'property_type': '建物のタイプ', 'accommodates': '定員',
            'bedrooms': '寝室数', 'beds': 'ベッド数',
            'bathrooms': 'バスルーム数',
            'bathrooms_text': 'バスルームの表記',
            'minimum_nights': '最低宿泊数',
            'availability_365': '予約可能日数',
            'number_of_reviews': 'レビュー数',
            'review_scores_rating': 'レビューの点数',
            'reviews_per_month': '月あたりレビュー数',
            'host_is_superhost': 'スーパーホスト',
            'host_listings_count': 'ホストの掲載数', 'price': '料金'}
raw = raw.rename(columns=ja_names)
raw.head()


各列の中身は次のとおりです。`NaN`と表示されたマスは、値が欠けている（空欄の）マスです。

| 日本語の名前 | 元の名前 | 中身 |
|---|---|---|
| `地区` | `neighbourhood` | 区や市の名前（例: `Ota Ku`は大田区） |
| `部屋のタイプ` | `room_type` | 貸切（`Entire home/apt`）・個室・相部屋など |
| `建物のタイプ` | `property_type` | マンションの一室・一軒家など |
| `定員` | `accommodates` | 泊まれる人数 |
| `寝室数`・`ベッド数`・`バスルーム数` | `bedrooms`・`beds`・`bathrooms` | 部屋の設備の数 |
| `バスルームの表記` | `bathrooms_text` | バスルーム数を文字で書いたもの（例: `1.5 baths`） |
| `最低宿泊数` | `minimum_nights` | 何泊から予約できるか |
| `予約可能日数` | `availability_365` | この先1年で予約を受け付ける日数 |
| `レビュー数`・`レビューの点数`・`月あたりレビュー数` | `number_of_reviews`ほか | 泊まった人の評価 |
| `スーパーホスト`・`ホストの掲載数` | `host_is_superhost`ほか | 貸し手（ホスト）の情報 |
| `料金` | `price` | 1泊の料金（円）。当てたい値 |


### 分析の地図：今日はここ

データ解析は「① データの理解と目標の設定 → ② 前処理（preprocessing）とデータ解析 → ③ 結果の解釈と目標との整合」の3つのフェーズを回ります。今日は ② の前処理が中心です。


In [ ]:
# @title 図：分析の地図
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.axis('off')
phases = ['① データの理解と\n目標の設定', '② 前処理と\nデータ解析', '③ 結果の解釈と\n目標との整合']
colors = ['#0066cc', '#2a9d8f', '#e63946']
for i, (x0, p, c) in enumerate(zip([0.17, 0.5, 0.83], phases, colors), start=1):
    on = i == 2
    ax.text(x0, 0.62, p, ha='center', va='center', fontsize=13 if on else 11, color='white',
            bbox=dict(boxstyle='round,pad=0.6', facecolor=c, alpha=0.95 if on else 0.3))
for a0, a1 in [(0.29, 0.365), (0.62, 0.695)]:
    ax.annotate('', xy=(a1, 0.62), xytext=(a0, 0.62),
                arrowprops=dict(arrowstyle='->', color='#555', lw=2))
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.show()


### 今日の道筋

前処理を進める順に、次の6つの小さな問いに答えていきます。

1. どの部屋を使う？（1〜2節）
2. なぜ欠けた？（3節）
3. どう補う？（4〜5節）
4. 目盛りをそろえると？（6節）
5. 同じ整え方を何度でも当てるには？（7節）
6. 1泊いくら？（8〜9節）


---


## 1. どの部屋を相場に入れるか


表が読めました。値段は、ほかの部屋の料金を手がかりに付けます。まず、どんな部屋が並んでいるかを確かめ、相場に入れる部屋を決めます。


### お手本：料金の形を見る

- 役割：料金の要約とヒストグラムを出す
- 入力：表`raw`
- 出力：料金の要約と、ヒストグラム

`describe()`は、件数・平均・標準偏差（standard deviation）・最小・四分位・最大をまとめて出すメソッドです。`plt.hist`の`range=(0, 300000)`は、図に描く範囲を0〜30万円に決める引数です。


In [ ]:
print(len(raw), '件')
print(raw['料金'].describe().round(0))
plt.hist(raw['料金'], bins=100, range=(0, 300000))
plt.xlabel('1泊の料金（円）')
plt.ylabel('部屋の数')
plt.show()


料金の中央値（50%の行）は約2万円なのに、最大は545万円です。ヒストグラムは2万円あたりに山が1つあり、右に長い尾が伸びています（30万円より先にも、545万円まで続きます）。

1泊数十万〜数百万円の部屋としては、一棟貸しの大きな家や、特定の日だけ値段を上げた掲載などが考えられます。ふつうの部屋とは別の市場にあると考えたほうがよさそうです。「新しいホストが、ふつうの部屋に値段を付ける」という今日の目標の手がかりになりません。

外すときの約束は2つです。外す理由を目標から説明できること。そして、どこで線を引いたかを書き残すことです。今日は「1泊10万円以下」に絞ります。外れるのは902件（2.8%）です。


### おさらい：条件で絞る

`raw[ ]`の中にTrue/Falseの並びを入れると、Trueの行だけが残ります。

```python
raw[raw['料金'] <= 100000]      # 10万円以下の部屋だけ
```


### 変えてみる：10万円以下の、貸切の部屋に絞る

- 役割：10万円以下の部屋を部屋のタイプごとに見てから、貸切の部屋に絞る
- 入力：表`raw`
- 出力：絞った表`df`と、その件数

`groupby('部屋のタイプ')`で部屋のタイプごとに分け、`agg(['median', 'count'])`で料金の中央値と件数を並べます。`reset_index(drop=True)`は、行の番号を0から振り直すメソッドです。


In [ ]:
df = raw[raw['料金'] <= 100000]                # 10万円以下の部屋だけ
print(df.groupby('部屋のタイプ')['料金'].agg(['median', 'count']))
df = df[df['部屋のタイプ'] == 'Entire home/apt']  # 貸切の部屋だけ
df = df[df['最低宿泊数'].notna()]   # 最低宿泊数の欠けた3件を外す
df = df.reset_index(drop=True)                   # 行の番号を振り直す
print(len(df), '件')


貸切（`Entire home/apt`）の中央値は約2万円、相部屋（`Shared room`）は3千円台です。同じ「1泊の料金」でも、別の商品です。あなたが出したいのは貸切の部屋なので、貸切だけに絞ります。目標に照らして使う行を決めるのも、前処理です。

`最低宿泊数`が欠けた部屋は、貸切の中に3件だけありました。理由は分かりませんが、2万6千件のうちの3件なので外します。`notna()`は、欠けていないマスをTrueにするメソッドです（3節でくわしく扱います）。残ったのは26,430件です。


### お手本：先に、訓練とテストに分ける

- 役割：部屋を、訓練8割・テスト2割に分ける
- 入力：表`df`
- 出力：訓練の表`tr`とテストの表`te`

第2回と同じ`train_test_split`です。このあとの前処理は、すべて訓練データだけを見て決めます。テストは、まだ見ていない部屋の代わりとして、最後の採点にだけ使います。


In [ ]:
tr, te = train_test_split(df, test_size=0.2, random_state=42)
print(len(tr), len(te))


訓練は21,144件、テストは5,286件です。


---


## 2. 何円外れるかを測る関数


相場に入れる部屋が決まりました。今日は、整えるたびに「まだ見ていない部屋で何円外れるか」を何度も測ります。同じ計算を毎回書かずに済むよう、関数にします。


### お手本：外れ幅の平均を関数にする

- 役割：実際の料金と予測から、外れ幅の平均（円）を出す
- 入力：実際の料金`y`と、予測`pred`
- 出力：外れ幅の平均（整数に丸めた円）

外れ幅の平均は、実際の料金$y_i$と予測$\hat{y}_i$の差の大きさを、$n$件で平均したものです。

$$
\text{外れ幅の平均} = \frac{1}{n}\sum_{i=1}^{n} \lvert y_i - \hat{y}_i \rvert
$$

`def 名前(引数):`で関数を作ります。引数は関数が受け取るもの（入力）、`return`の後ろが関数が返すもの（戻り値、出力）です。これまで「役割・入力・出力」と読んできたかたまりに、名前を付けて何度も呼べるようにしたものが関数です。関数の中身は、字下げした行です。`np.abs`は絶対値、`.mean()`は平均、`round`は四捨五入です。


In [ ]:
def mae(y, pred):          # 外れ幅の平均
    return round(np.abs(y - pred).mean())


このセルを実行しても、何も表示されません。関数を作っただけだからです。使うには、名前にかっこを付けて引数を渡します（呼び出す）。


### 変えてみる：関数を呼び出す

- 役割：どの部屋も「訓練の料金の中央値」と答えたときの、テストでの外れ幅を測る
- 入力：訓練とテストの料金
- 出力：中央値と、外れ幅の平均

何も工夫しない答えです。これを、このあと整えた式と比べる基準にします。


In [ ]:
base = np.median(tr['料金'])           # 訓練の中央値で答える
print(base)
print(mae(te['料金'], base), '円')     # 関数を呼び出す


どの部屋も約2万円と答えると、外れ幅は平均10,860円です。これが基準です。


### 式に入れようとすると、止まる

第2回のように、数の列を手がかりに線形回帰の式を作ろうとすると、次のエラーで止まります（このノートでは実行しません）。

```python
cols = ['定員', '寝室数', 'ベッド数', 'バスルーム数', '最低宿泊数',
        '予約可能日数', 'レビュー数', 'ホストの掲載数']
LinearRegression().fit(tr[cols], tr['料金'])
```

```
ValueError: Input X contains NaN.
```

いちばん下の行を読むと「手がかりの表`X`に`NaN`（欠けた値）が入っている」とあります。欠けたマスがあると、式は作れません。欠けをどうにかするのが、次の仕事です。


In [ ]:
cols = ['定員', '寝室数', 'ベッド数', 'バスルーム数', '最低宿泊数',
        '予約可能日数', 'レビュー数', 'ホストの掲載数']


手がかりにする8本の列の名前を、`cols`にしまっておきます。


---


## 3. なぜ欠けたのか


欠けた値があると、式が作れませんでした。すぐに埋めたくなりますが、その前に、どの列がどれだけ、なぜ欠けたのかを確かめます。欠けた理由によって、扱い方が変わるからです。


### TODO①：列ごとの欠測の数と割合を表示する

`tr.isna()`は、各マスが欠けているかをTrue/Falseで表します。`.sum()`で列ごとにTrueの数を、`.mean()`でTrueの割合を出せます（Trueは1、Falseは0として数えます）。欠測の割合は、欠けたマスの数を行の数$n$で割ったものです。訓練データ`tr`について、欠測の数と割合（%）を表示してください。


In [ ]:
# TODO: 訓練データ tr の、列ごとの欠測の数と割合（%）を表示する

# 解答例①：列ごとの欠測の数と割合
print(pd.DataFrame({'欠測の数': tr.isna().sum(),
                    '割合': (tr.isna().mean() * 100).round(1)}))


欠けているのは5つの列です。バスルーム数1,322件（6.3%）、寝室数1,025件（4.8%）、ベッド数838件（4.0%）、そしてレビューの点数と月あたりレビュー数が2,310件（10.9%）ずつです。地区や料金などの列は欠けていません。

欠け方には、大きく3つの形があります。

- でたらめに欠ける：どの部屋でも同じように欠ける。入力し忘れなど
- ほかの列しだいで欠ける：小さい部屋ほど欠けやすい、のように、ほかの列で欠けやすさが変わる
- その値自身が理由で欠ける：値があり得ないから、値が都合の悪いものだから欠ける

（英語の名前で、MCAR・MAR・MNARと呼ばれます。）どの形かによって、扱い方が変わります。順に見ていきます。


### お手本：レビューの点数は、なぜ欠けた？

- 役割：レビューが0件の部屋の数と、レビューの点数が欠けた部屋の数を比べる
- 入力：訓練の表`tr`
- 出力：3つの件数

`&`は「かつ」です。2つの条件の両方がTrueの行を数えます。


In [ ]:
no_review = tr['レビュー数'] == 0              # レビューが0件の部屋
no_score = tr['レビューの点数'].isna()        # 点数が欠けた部屋
print(no_review.sum(), no_score.sum())
print((no_review & no_score).sum())          # 両方にあてはまる部屋


3つとも2,310件です。点数が欠けた部屋は、ぴったりレビューが0件の部屋でした。まだ誰も泊まっていないので、点数が付きようがないのです。値そのものが存在しない欠けです。

こうした欠けを平均点で埋めると、「まだ誰も泊まっていない」という情報が消え、泊まった人のいない部屋に、ありもしない点数が付きます。今日は、点数の列も月あたりレビュー数の列も使いません（レビューが0件かどうかは、`レビュー数`の列が持っています）。


### お手本：欠けたバスルーム数を見る

- 役割：バスルーム数が欠けた部屋を取り出し、文字の列`バスルームの表記`に何と書いてあるかを数える
- 入力：訓練の表`tr`
- 出力：欠けた件数と、表記の中身の多い順

`value_counts()`は、中身ごとの件数を多い順に並べるメソッドです。


In [ ]:
miss = tr['バスルーム数'].isna()      # 欠けた部屋は True
print(miss.sum(), '件')
print(tr[miss]['バスルームの表記'].value_counts())


欠けた1,322件のどれにも、`1 bath`・`1.5 baths`・`Half-bath`のように表記が書いてあります。数の列だけが空欄になっていたのです。それなら、推測するより、文字から本当の値を戻すほうが確かです。


### 変えてみる：文字1つを数1つに直す関数

- 役割：`'1.5 baths'`のような表記を1つ受け取り、バスルームの数を返す
- 入力：表記の文字`text`
- 出力：数（`1.5 baths`なら1.5）

`'1.5 baths'.split(' ')`は、空白で切り分けて`['1.5', 'baths']`にします。`[0]`で最初の`'1.5'`を取り、`float( )`で小数の数1.5にします。`'Half-bath'`（シャワーのない洗面所など、半分のバスルーム）だけは数が書いていないので、`if`で0.5を返します。`return`したところで関数は終わるので、`Half-bath`のときは下の行まで進みません。


In [ ]:
def bath_num(text):            # '1.5 baths' → 1.5
    if text == 'Half-bath':
        return 0.5
    return float(text.split(' ')[0])

print(bath_num('1.5 baths'), bath_num('Half-bath'))  # 確かめる


作った関数は、まず1回呼んで確かめます。`1.5`と`0.5`が出れば正しく動いています。


### 変えてみる：表の欠けを、文字から戻す関数

- 役割：表`d`の、欠けたバスルーム数を、表記から読み取った数で埋める
- 入力：表`d`（訓練でもテストでも新しい部屋でもよい）
- 出力：バスルーム数の欠けが埋まった表

いま作った`bath_num`を、表の列全体に当てます。

欠けを補う処理は、どのやり方でも次の1本の式で書けます。$x_i$は$i$番目の部屋の値、$\hat{x}_i$は欠けたときに入れる値（埋める値）、$\tilde{x}_i$は補ったあとの値です。

$$
\tilde{x}_i = \begin{cases} x_i & (\text{値がある}) \\ \hat{x}_i & (\text{欠けている}) \end{cases}
$$

ここでの$\hat{x}_i$は、表記から読んだ数です。コードの`fillna(from_text)`が、この式そのものです。値があればそのまま、欠けていれば`from_text`の値にします。

- `d.copy()`：受け取った表の写しを作る。関数の外の表を、知らないうちに書き換えないため
- `map(bath_num)`：列の中身を1つずつ`bath_num`に通した列を返す
- `fillna(列)`：欠けたマスだけを、同じ行の値で埋める。欠けていないマスはそのまま


In [ ]:
def fill_bath(d):              # 欠けたバスルーム数を、文字から戻す
    d = d.copy()
    from_text = d['バスルームの表記'].map(bath_num)
    d['バスルーム数'] = d['バスルーム数'].fillna(from_text)
    return d

tr = fill_bath(tr)
te = fill_bath(te)
print(tr['バスルーム数'].isna().sum())
print(te['バスルーム数'].isna().sum())


戻したあとの欠けは、訓練もテストも0件です。その部屋の表記だけを読む処理なので、訓練・テストのどちらに当ててもかまいません。

`fill_bath`の形を覚えておいてください。「埋める値の列を作り（`from_text`）、`fillna`で欠けたマスだけ差し替える」。このあと、推測で補うときも同じ形を使います。


### 変えてみる：寝室数とベッド数は、どんな部屋で欠けた？

- 役割：寝室数が欠けた部屋と欠けていない部屋で、定員と料金の中央値を比べる
- 入力：訓練の表`tr`
- 出力：2行の表と、寝室数が0の部屋の数

`groupby`に、True/Falseの並びを渡すと、欠けた部屋（True）と欠けていない部屋（False）に分けて集計できます。


In [ ]:
miss_bed = tr['寝室数'].isna()           # 寝室数が欠けた部屋は True
print(tr.groupby(miss_bed)[['定員', '料金']].median())
print('寝室数が0の部屋:', (tr['寝室数'] == 0).sum(), '件')


寝室数が欠けた部屋（True）は、定員の中央値が2人、料金が15,948円です。欠けていない部屋（False）は4人・20,358円です。欠けた部屋は、小さくて安い部屋に偏っています。ベッド数でも同じように調べると、欠けた部屋の定員の中央値は3人で、欠けていない部屋の4人より小さくなります。「ほかの列（定員）しだいで欠ける」形です。


### 問い：寝室数0の部屋が、1件もない

寝室数が0の部屋は、1件もありません。寝室数が欠けた部屋は、定員2人の小さい部屋に偏っていました。この欠けは何を意味していそうでしょうか。予想を下に書いてください。


（ここに予想を書く）


ワンルーム（寝室として区切られた部屋がない）を、「0」ではなく欠けとして登録した部屋が混ざっているのかもしれません。もしそうなら、「その値自身（0）が理由で欠ける」形も混ざっています。

どちらにしても、寝室数とベッド数は、料金の手がかりとして使いたい列です。欠けたまま外すと、小さい部屋ばかりが減ってしまいます。次の節で、どう補うのがよいかを確かめます。


---


## 4. 補い方を採点する


寝室数とベッド数の欠けは、補って使いたい列でした。ただ、本当の値が分からない欠けでは、補った値が当たったかどうかを確かめられません。そこで、答えの分かる欠けを自分で作り、補い方を採点します。

ここで使うのは、練習用に作ったデータです。ベッド数が分かっている部屋から、わざと値を隠します。実際のデータの欠けは4%ほどですが、違いが見えやすいように、欠けの率を上げて隠します。


### お手本：答えの分かる欠けを作る関数

- 役割：表`d`の列`col`の値を、定員の小さい部屋ほど多く隠す
- 入力：表`d`、隠す列`col`、欠けの率の目安`rate`、乱数（random number）の種`seed`
- 出力：値を隠した表

これは練習用の道具です。役割・入力・出力と、次の1点だけ押さえれば、中身は読み飛ばしてかまいません。

`seed=0`のように、引数に`=`で値を書いておくと、呼び出すときに省いた場合はその値が使われます（既定値）。下の`make_missing(full, 'ベッド数', 0.2)`は、`seed`を書いていないので`seed=0`で動きます。

中身が気になる人へ：部屋$i$を隠す確率を$p_i$、欠けの率の目安を$r$（`rate`）とすると、

$$
p_i = \frac{4r}{\text{定員}_i}
$$

です。`rng.random(len(d))`で0〜1の乱数を部屋の数だけ出し、それが$p_i$（`rate * 4 / d['定員']`）より小さい部屋を隠します。隠す確率は、定員4人の部屋で`rate`、2人でその2倍、8人で半分です。`d.loc[hide, col] = np.nan`は、`hide`がTrueの行の列`col`のマスを、欠けた値`np.nan`で書き換えます。


In [ ]:
def make_missing(d, col, rate, seed=0):
    d = d.copy()
    rng = np.random.default_rng(seed)
    hide = rng.random(len(d)) < rate * 4 / d['定員']
    d.loc[hide, col] = np.nan
    return d

full = tr[tr['ベッド数'].notna()]    # ベッド数の分かる部屋
practice = make_missing(full, 'ベッド数', 0.2)
miss = practice['ベッド数'].isna()
print(len(full), '件のうち', miss.sum(), '件を隠した')
print(round(miss.mean() * 100, 1), '%')


ベッド数が分かっている20,306件のうち、4,631件（22.8%）を隠しました。隠した部屋の本当のベッド数は、`full`に残っています。


### お手本：平均で埋めて、採点する

- 役割：隠したベッド数を、残った部屋のベッド数の平均で埋め、本当の値と比べる
- 入力：練習用の表`practice`と、本当の値の表`full`
- 出力：埋めた値と、隠した部屋での外れの平均

平均で埋めるのは、補完の式の$\hat{x}_i$に、値の残った部屋（$O$）の平均を入れることです。部屋によらず、同じ値になります。

$$
\hat{x}_i = \bar{x} = \frac{1}{|O|}\sum_{k \in O} x_k
$$

採点は、隠した部屋（$M$）だけで、埋めた値$\tilde{x}_i$と本当の値$x_i$の差の大きさを平均します。2節の外れ幅の平均と同じ形です。

$$
\frac{1}{|M|}\sum_{i \in M} \lvert \tilde{x}_i - x_i \rvert
$$

`fillna(値)`は、欠けたマスを1つの値で置き換えます。`[miss]`で、隠した部屋だけを取り出して外れを測ります。


In [ ]:
avg = practice['ベッド数'].mean()               # 残った部屋の平均
by_mean = practice['ベッド数'].fillna(avg)      # 欠けを平均で埋める
truth = full['ベッド数']                        # 本当の値
print('埋めた値:', round(avg, 2))
print('外れ:', round(np.abs(by_mean - truth)[miss].mean(), 2), '台')


平均3.27台で埋めると、隠した部屋の本当のベッド数から、平均1.63台外れます。定員2人の部屋にも8人の部屋にも、同じ3.27台が入るからです。


In [ ]:
# @title 図：平均で埋めると、どの部屋にも同じ値が入る
rng_j = np.random.default_rng(1)
jit = rng_j.uniform(-0.3, 0.3, len(full))
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
axes[0].scatter(full['定員'] + jit, truth + jit[::-1], s=4, alpha=0.15, color='#888888')
axes[0].scatter((full['定員'] + jit)[miss], (truth + jit[::-1])[miss], s=4, alpha=0.3, color='#2a9d8f')
axes[0].set_title('本当の値（緑が隠した部屋）')
axes[1].scatter((full['定員'] + jit)[~miss], (truth + jit[::-1])[~miss], s=4, alpha=0.15, color='#888888')
axes[1].scatter((full['定員'] + jit)[miss], by_mean[miss], s=4, alpha=0.3, color='#e63946')
axes[1].set_title('平均で埋めた（赤）')
for ax in axes:
    ax.set_xlabel('定員'); ax.set_xlim(0, 17); ax.set_ylim(0, 17)
axes[0].set_ylabel('ベッド数')
plt.tight_layout()
plt.show()


左が本当の値、右が平均で埋めた値です（点が重ならないよう、少しずらして描いています）。本当は、定員が多いほどベッド数も多い右上がりの関係があります。平均で埋めると、隠した部屋（赤）は定員に関係なく3.27台の高さに一直線に並びます。

その結果、定員とベッド数の相関係数（2つの列の直線的な関係の強さ。−1〜1）も弱まります。


In [ ]:
print('本当の値  :', round(full['定員'].corr(truth), 3))
print('平均で埋めた:', round(full['定員'].corr(by_mean), 3))


本当は0.848ある関係が、平均で埋めると0.762まで弱まります。埋めた値が、定員の情報を無視しているからです。


### お手本：定員から推測して補う

- 役割：値の分かる部屋で「定員からベッド数を当てる式」を作り、欠けた部屋をその推測で埋める
- 入力：練習用の表`practice`
- 出力：推測で埋めたベッド数`by_reg`

ベッド数は、定員と強く関係していました。それなら、第2回の線形回帰を、料金ではなくベッド数を当てるのに使えばよいはずです。

$$
\widehat{\text{ベッド数}} = a \times \text{定員} + b
$$

$a$と$b$は、ベッド数の分かる部屋で決めます。やり方は、3節の`fill_bath`と同じ形です。埋める値の列（ここでは`推測`）を作り、`fillna`で欠けたマスだけ差し替えます。`practice['推測'] = ...`のように、表に新しい列名で値を入れると、列が1本増えます。


In [ ]:
ok = practice[practice['ベッド数'].notna()]          # 値の分かる部屋
m_bed = LinearRegression().fit(ok[['定員']], ok['ベッド数'])
practice['推測'] = m_bed.predict(practice[['定員']])
by_reg = practice['ベッド数'].fillna(practice['推測'])
print('外れ:', round(np.abs(by_reg - truth)[miss].mean(), 2), '台')
print('相関:', round(full['定員'].corr(by_reg), 3))


定員から推測すると、外れは0.58台です。平均で埋めた1.63台の、3分の1ほどに縮みました。相関も0.863と、本当の0.848に近くなります（少し強めに出る理由は、発展で扱います）。


In [ ]:
# @title 図：本当の値・平均で埋めた・定員から推測した
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
panels = [('本当の値', truth + jit[::-1], '#2a9d8f'),
          ('平均で埋めた', by_mean, '#e63946'),
          ('定員から推測した', by_reg, '#0066cc')]
for ax, (title, val, c) in zip(axes, panels):
    ax.scatter((full['定員'] + jit)[~miss], (truth + jit[::-1])[~miss], s=4, alpha=0.15, color='#888888')
    ax.scatter((full['定員'] + jit)[miss], val[miss], s=4, alpha=0.3, color=c)
    ax.set_title(title); ax.set_xlabel('定員'); ax.set_xlim(0, 17); ax.set_ylim(0, 17)
axes[0].set_ylabel('ベッド数')
plt.tight_layout()
plt.show()


推測した値（右、青）は、定員が増えるほど上がる1本の線に乗ります。平均で埋めた値（中央、赤）のように、定員を無視することはありません。


### 変えてみる：採点を関数にまとめる

- 役割：欠けの率`rate`で欠けを作り、平均で埋めた場合と推測で補った場合を採点して表示する
- 入力：欠けの率の目安`rate`
- 出力：なし（欠けの率・埋めた平均・2つの外れと相関を表示する）

欠けの率を変えて、何度も採点し直したくなりました。上の3つのお手本のセルを、そのまま字下げして`def`の下に並べ、`rate`だけを引数にします。こうすると、`grade(0.4)`と1行書くだけで、欠けの率0.4でやり直せます。

関数の中で作った名前（`practice`や`miss`など）は、その関数の中だけのものです。関数の外の`practice`や`miss`は書き換わりません。


In [ ]:
def grade(rate):           # 欠けの率 rate で、2つの補い方を採点する
    practice = make_missing(full, 'ベッド数', rate)
    miss = practice['ベッド数'].isna()
    avg = practice['ベッド数'].mean()
    by_mean = practice['ベッド数'].fillna(avg)
    ok = practice[practice['ベッド数'].notna()]
    m_bed = LinearRegression().fit(ok[['定員']], ok['ベッド数'])
    practice['推測'] = m_bed.predict(practice[['定員']])
    by_reg = practice['ベッド数'].fillna(practice['推測'])
    print('欠けの率', round(miss.mean() * 100, 1), '%',
          ' 埋めた平均', round(avg, 2))
    print(' 平均', round(np.abs(by_mean - truth)[miss].mean(), 2),
          ' 相関', round(full['定員'].corr(by_mean), 3))
    print(' 推測', round(np.abs(by_reg - truth)[miss].mean(), 2),
          ' 相関', round(full['定員'].corr(by_reg), 3))

grade(0.2)


上のお手本と同じ数（外れ1.63台と0.58台）が出れば、正しくまとまっています。


### TODO②：欠けの率を変えて、採点し直す

`grade`を、`rate`を0.05にして1回、0.4にして1回呼んでください。平均で埋めた値・外れ・相関が、欠けの率でどう変わるかを確かめます。


In [ ]:
# TODO: grade を rate=0.05 と rate=0.4 で呼ぶ

# 解答例②：rate を 0.05 と 0.4 にして呼ぶ
grade(0.05)
grade(0.4)


| 欠けの率の目安`rate` | 欠けた割合 | 埋めた平均 | 平均で埋めた外れ・相関 | 推測した外れ・相関 |
|---|---|---|---|---|
| 0.05 | 5.8% | 3.10台 | 1.51台・0.828 | 0.58台・0.852 |
| 0.2 | 22.8% | 3.27台 | 1.63台・0.762 | 0.58台・0.863 |
| 0.4 | 45.5% | 3.70台 | 1.90台・0.650 | 0.59台・0.882 |

欠けが増えるほど、平均で埋めた値は大きく外れ、相関は本当の0.848から離れていきます。定員から推測した外れは、欠けの率によらず0.58台前後のままです。


### 問い：埋める平均が、だんだん大きくなる

欠けを増やすと、平均で埋める値そのものが3.10台 → 3.27台 → 3.70台と大きくなりました。全部の部屋の本当の平均は3.05台です。なぜ、欠けが増えるほど平均が大きくなるのでしょうか。下に書いてください。


（ここに予想を書く）


小さい部屋ほど多く隠したからです。隠した部屋の本当の平均は2.26台ほどで、残った部屋は大きめの部屋に偏ります。残った部屋の平均は、全体の平均より大きくなり、それで小さい部屋の欠けを埋めることになります。

「ほかの列しだいで欠ける」ときは、残った部屋の平均そのものがずれます。平均で埋めてはいけない、いちばんの理由です。欠けの率が低くても、外れは1.5台あり、平均で埋めてよいことにはなりません。


---


## 5. 実データの欠けを補う


練習用の欠けで、平均で埋めるより、定員から推測するほうがずっと当たると分かりました。この推測を、本当の欠けに当てます。

今度は、埋める表が訓練とテストの2つ、埋める列が寝室数とベッド数の2本あります。4節のお手本の4行を、組み合わせの数だけ書くことになります。そこで、関数にします。


### お手本：推測で補う関数

- 役割：表`d`の列`col`の欠けを、定員から推測した値で埋める
- 入力：推測の式を作る表`base`、埋める表`d`、列の名前`col`
- 出力：欠けを埋めた表

中身は、4節のお手本と同じです。違うのは、表を2つ受け取ることです。推測の式は`base`で作り、欠けを埋めるのは`d`です。テストを埋めるときも、推測の式は訓練で作りたいからです。テストは、まだ見ていない部屋の代わりです。


In [ ]:
def fill_reg(base, d, col):     # 欠けを、定員からの推測で埋める
    ok = base[base[col].notna()]                  # 値の分かる部屋
    m = LinearRegression().fit(ok[['定員']], ok[col])
    d = d.copy()
    d['推測'] = m.predict(d[['定員']])            # どの部屋も推測
    d[col] = d[col].fillna(d['推測'])             # 欠けだけ差し替え
    return d

by_reg2 = fill_reg(practice, practice, 'ベッド数')['ベッド数']
print('外れ:', round(np.abs(by_reg2 - truth)[miss].mean(), 2), '台')


まず、4節の練習用の表で1回呼んで確かめます。4節と同じ0.58台が出れば、正しくまとまっています。


### 変えてみる：訓練とテストを補い、料金の式を作る

- 役割：訓練で作った推測の式で、訓練とテストの寝室数・ベッド数を補い、料金の線形回帰の式を作って測る
- 入力：訓練の表`tr`とテストの表`te`
- 出力：補った表`tr_f`・`te_f`と、テストでの外れ幅

最初の引数は、4行とも訓練`tr`です。


In [ ]:
tr_f = fill_reg(tr, tr, '寝室数')
tr_f = fill_reg(tr, tr_f, 'ベッド数')
te_f = fill_reg(tr, te, '寝室数')
te_f = fill_reg(tr, te_f, 'ベッド数')
print(tr_f[cols].isna().sum().sum(), te_f[cols].isna().sum().sum())

model = LinearRegression().fit(tr_f[cols], tr_f['料金'])
print(mae(te_f['料金'], model.predict(te_f[cols])), '円')


欠けは訓練もテストも0件になり、式が作れました（`.sum().sum()`は、列ごとの数をさらに足して1つの数にします）。外れ幅は8,907円で、基準の10,860円から約2千円縮みました。


### 変えてみる：平均で埋めた式と比べる

- 役割：寝室数とベッド数を訓練の平均で埋めた式を作り、テストの外れ幅を比べる
- 入力：`tr`・`te`
- 出力：2つの式の外れ幅

`for col in [...]`は、第2回の`for`文と同じで、並びの中身を1つずつ`col`に入れながら、字下げした行を繰り返します。


In [ ]:
tr_m = tr.copy()
te_m = te.copy()
for col in ['寝室数', 'ベッド数']:
    tr_m[col] = tr[col].fillna(tr[col].mean())    # 訓練の平均で埋める
    te_m[col] = te[col].fillna(tr[col].mean())
model_m = LinearRegression().fit(tr_m[cols], tr_m['料金'])
pred_f = model.predict(te_f[cols])
pred_m = model_m.predict(te_m[cols])
print('推測', mae(te['料金'], pred_f))
print('平均', mae(te['料金'], pred_m))


### 問い：全体では、ほとんど同じ

テストの全部の部屋で見ると、外れ幅は推測で8,907円、平均で8,911円と、4円しか違いません。それなら、平均で埋めてもよいのでしょうか。下に書いてください。


（ここに予想を書く）


### 変えてみる：欠けのあった部屋だけで比べる

- 役割：テストのうち、寝室数かベッド数が欠けていた部屋だけで、2つの式の外れ幅を比べる
- 入力：上のセルの`pred_f`・`pred_m`
- 出力：欠けのあった部屋の数と、2つの外れ幅

`|`は「または」です。寝室数かベッド数のどちらかが欠けていた部屋がTrueになります。


In [ ]:
had = te['寝室数'].isna() | te['ベッド数'].isna()   # 欠けのあった部屋
print(had.sum(), '件')
print('推測', mae(te['料金'][had], pred_f[had]),
      ' 平均', mae(te['料金'][had], pred_m[had]))


テストの5,286件のうち、寝室数かベッド数が欠けていたのは448件（8%）だけです。残りの9割は補う必要がなく、どちらのやり方でも同じ予測になります。だから全体の平均では、差が薄まって見えません。

欠けのあった448件だけで比べると、推測で7,759円、平均で8,060円と、300円ほど差が出ます。補い方の差は、補った部屋に出るのです。そして4節で見たとおり、欠けが増えるほど、平均で埋めた害は大きくなります。全体の点数だけを見て「平均でも同じ」と判断せず、補った値そのものを確かめることが大切です。


---


## 6. 目盛りをそろえる


欠けが補えて、料金の式ができました。8本の手がかりの列は、目盛りがばらばらです。予約可能日数は0〜365日、バスルーム数は0〜4室ほど。列どうしを同じ物差しで比べられるよう、目盛りをそろえます。


### お手本：列ごとのばらつきを見る

- 役割：訓練の8本の列の、標準偏差を並べる
- 入力：補った訓練の表`tr_f`
- 出力：列ごとの標準偏差

標準偏差は、値が平均からどれくらい散らばっているかの目安です。平均を$\bar{x}$として、

$$
s = \sqrt{\frac{1}{n-1}\sum_{i=1}^{n} (x_i - \bar{x})^2}
$$

平均からの外れを二乗して平均し、平方根で元の単位に戻したものです（pandasの`.std()`は$n-1$で割ります）。


In [ ]:
X_tr = tr_f[cols]
X_te = te_f[cols]
print(X_tr.std().round(1))


予約可能日数は92、ホストの掲載数は75、レビュー数は67日・件ほど散らばる一方、寝室数は0.9、バスルーム数は0.4しか散らばりません。


### お手本：標準化の関数

- 役割：表の各列を、決められた平均と標準偏差で標準化する
- 入力：表`X`、列ごとの平均`mean`、列ごとの標準偏差`std`
- 出力：標準化した表

標準化は、各列から平均を引き、標準偏差で割ることです。$j$番目の列の平均を$\bar{x}_j$、標準偏差を$s_j$とすると、

$$
z_{ij} = \frac{x_{ij} - \bar{x}_j}{s_j}
$$

どの列も「平均0・標準偏差1」の目盛りにそろい、値は「平均から標準偏差いくつぶん離れているか」を表します。平均と標準偏差は、訓練データだけで決めます。`X_tr.mean()`のように表に`.mean()`を付けると、列ごとの平均が並んで返ります。


In [ ]:
def standardize(X, mean, std):
    return (X - mean) / std

mean_tr = X_tr.mean()            # 列ごとの平均（訓練だけ）
std_tr = X_tr.std()              # 列ごとの標準偏差（訓練だけ）
Z_tr = standardize(X_tr, mean_tr, std_tr)
print(Z_tr.mean().round(2))      # どの列も平均0になったか
print(Z_tr.std().round(2))       # どの列も標準偏差1になったか


どの列も平均0・標準偏差1にそろいました。表示の`-0.0`は、ごく小さなマイナスの値を丸めたもので、0と同じです。


### TODO③：テストも、訓練の物差しで標準化する

テストも、訓練の平均`mean_tr`と標準偏差`std_tr`で標準化して`Z_te`を作り、列ごとの平均を表示してください。


In [ ]:
# TODO: Z_te を作り、列ごとの平均を表示する

# 解答例③：テストを訓練の物差しで標準化する
Z_te = standardize(X_te, mean_tr, std_tr)
print(Z_te.mean().round(2))


テストの列ごとの平均は、0に近いものの、ぴったり0にはなりません（たとえば0.01や−0.02）。テストの平均ではなく、訓練の平均を引いたからです。ぴったり0にならないことが、訓練の物差しを使った証拠です。


### 変えてみる：標準化してから式を作る

- 役割：標準化した訓練で線形回帰の式を作り、テストでの外れ幅と、係数を並べる
- 入力：`Z_tr`と、テストの`X_te`、5節の`model`
- 出力：外れ幅と、標準化する前と後の係数の表


In [ ]:
model_z = LinearRegression().fit(Z_tr, tr_f['料金'])
pred_z = model_z.predict(standardize(X_te, mean_tr, std_tr))
print(mae(te_f['料金'], pred_z), '円')
coef = pd.DataFrame({'そのまま': model.coef_,
                     '標準化後': model_z.coef_}, index=cols)
print(coef.round(0))


外れ幅は8,907円で、標準化する前と1円も変わりません。線形回帰は、列ごとの係数が目盛りを吸収するからです。式で書くと、$z_j = (x_j - \bar{x}_j)/s_j$を入れても

$$
\hat{y} = b_0 + \sum_j b_j x_j = b'_0 + \sum_j b'_j z_j, \qquad b'_j = b_j\,s_j
$$

となり、係数が標準偏差$s_j$倍になるだけです。たとえば予約可能日数を「日」で測っても「年」で測っても、係数が365倍違うだけで、予測は同じになります。

変わるのは係数です。


そのままの係数では、バスルーム数が1室あたり3,537円と、定員の1人あたり1,507円より大きく見えます。でも、バスルーム数は0.4室しか散らばらない列でした。標準化すると、どの係数も「その列が標準偏差1つぶん増えたら、料金が何円変わるか」という同じ目盛りになり、定員4,395円、寝室数2,959円、バスルーム数1,414円の順に並びます。

予測には効かなくても、列どうしを同じ物差しで比べたいときには、目盛りをそろえておく必要があります。そして、その物差し（平均と標準偏差）も、訓練データだけで決めます。


---


## 7. 同じ整え方を何度でも


ここまでで、整え方がそろいました。バスルーム数は文字から戻し、寝室数とベッド数は訓練で作った式で推測して補います。この整え方は、訓練にも、テストにも、そしてあなたの部屋にも、同じように当てる必要があります。毎回同じ行を書き写すと、どこかでずれます。1つの関数にまとめます。


### お手本：整え方を1つの関数にまとめる

- 役割：補い方を`base`で決め、表`d`を整えて、手がかりの8本の列を返す
- 入力：補い方を決める表`base`、整える表`d`
- 出力：欠けのない8本の列の表

関数の中で、3節の`fill_bath`と5節の`fill_reg`を呼んでいます。小さな関数を組み合わせて、大きな処理を作るのです。


In [ ]:
def prep(base, d):           # 補い方は base で決め、d を整える
    d = fill_bath(d)
    d = fill_reg(base, d, '寝室数')
    d = fill_reg(base, d, 'ベッド数')
    return d[cols]

model = LinearRegression().fit(prep(tr, tr), tr['料金'])
err = mae(te['料金'], model.predict(prep(tr, te)))
print('訓練で決めた:', err, '円')


5節と同じ8,907円です。5節の8行が、`prep`の1行ずつになりました。


### 変えてみる：全データで補い方を決めると

`prep`の最初の引数を、訓練`tr`から全データ`df`に変えるだけです。テストの部屋も含めて、推測の式を作ることになります。


In [ ]:
model_l = LinearRegression().fit(prep(df, tr), tr['料金'])
err_l = mae(te['料金'], model_l.predict(prep(df, te)))
print('全データで決めた:', err_l, '円')


全データで決めても8,907円で、差は出ませんでした。2万件以上もあると、テストの5千件を混ぜても、定員から寝室数を当てる式はほとんど変わらないからです。

それでも、補い方は訓練だけで決めます。テストの情報が前処理を通して式に漏れることを、第2回で見たリーク（leakage）と同じく避けるためです。データが少ないときや、テストの部屋だけ傾向が違うときには、差が大きく出ます。差が出るかどうかを確かめてから順番を守るのではなく、いつも訓練だけで決める習慣にします。`prep(base, d)`のように「決める表」と「整える表」を分けて受け取る関数にしておくと、この順番を間違えにくくなります。


---


## 8. 1泊いくら？


整え方を`prep`1つにまとめました。いよいよ、あなたの部屋に値段を付けます。


### お手本：あなたの部屋を値付けする

- 役割：あなたの部屋の条件を1行の表にし、`prep`で整えて、料金の式で値付けする
- 入力：あなたの部屋の表`my_room`
- 出力：1泊の値段（円）

例えば「定員4人、寝室2・ベッド2・バスルーム1」の貸切の部屋です。レビューはまだ0件、最低宿泊数は2泊、この先1年で200日予約を受け付け、ホストの掲載はこの1件だけとします。新しい部屋にも、訓練で決めた同じ整え方を当てます。`prep`の2つ目の引数に、新しい部屋を渡すだけです。`pd.DataFrame({列名: [値], ...})`で、1行の表を作れます。


In [ ]:
my_room = pd.DataFrame({'定員': [4], '寝室数': [2], 'ベッド数': [2],
                        'バスルーム数': [1],
                        'バスルームの表記': ['1 bath'],
                        '最低宿泊数': [2], '予約可能日数': [200],
                        'レビュー数': [0], 'ホストの掲載数': [1]})
price = model.predict(prep(tr, my_room))[0]
print(round(price), '円')


答えは24,830円です。予測は並びで返るので、`[0]`で1つ目を取り出しています。


### TODO④：あなたの番。自分の部屋を値付けする

定員・寝室数などを自分で決めて、1泊の値付けの目安を出してください。寝室数が分からない部屋なら、`np.nan`（欠けた値）を入れてもかまいません。`prep`が、訓練で作った式で補ってくれます。


In [ ]:
# TODO: my_room の条件を変えた部屋を作り、prep と model で値付けする

# 解答例④：定員2人・寝室1・ベッド1の部屋と、寝室数の分からない部屋
room2 = my_room.assign(定員=2, 寝室数=1, ベッド数=1)
print(round(model.predict(prep(tr, room2))[0]), '円')
room3 = my_room.assign(寝室数=np.nan)
print(round(model.predict(prep(tr, room3))[0]), '円')


定員2人・寝室1・ベッド1なら約1万8千円、定員4人で寝室数を`np.nan`にすると、寝室数を定員から推測して約2万3千円になります。


---


## 9. 外す線で、値段は変わるか


あなたの部屋の値段が出ました。ただ、この値段は、1節で「1泊10万円以下」に線を引いた結果です。線を変えたら、値段はどれくらい変わるのでしょうか。


### お手本：外す線を引数にした関数

- 役割：`cut`円以下の貸切の部屋で、絞る・分ける・整えて式を作る・値付けして採点する、を通して行う
- 入力：外す線`cut`（既定値は10万円）
- 出力：なし（値段と外れ幅を表示する）

長く見えますが、新しい書き方はありません。今日これまでに書いたセルを、4つのかたまりで並べただけです。

1. 絞る（3行）：1節と同じ。10万円の代わりに`cut`を使う
2. 分ける（1行）：1節と同じ`train_test_split`
3. 整えて式を作る（1行）：7節の`prep`と同じ
4. 値付けして採点する（3行）：8節の値付けと、2節の`mae`

`cut=100000`のように既定値を書いておくと、`price_for()`とかっこだけで呼べば10万円で、`price_for(50000)`と書けば5万円で動きます。


In [ ]:
def price_for(cut=100000):
    d = raw[raw['料金'] <= cut]
    d = d[d['部屋のタイプ'] == 'Entire home/apt']
    d = d[d['最低宿泊数'].notna()]
    tr2, te2 = train_test_split(d, test_size=0.2, random_state=42)
    m = LinearRegression().fit(prep(tr2, tr2), tr2['料金'])
    p = m.predict(prep(tr2, my_room))[0]
    err = mae(te2['料金'], m.predict(prep(tr2, te2)))
    print(cut, '円まで:', round(p), '円  外れ幅', err, '円')

price_for()


既定値の10万円では、8節と同じ24,830円・8,907円です。


### 変えてみる：線を動かす

`for`文で、外す線を3万円から1千万円（実質、外さない）まで変えて呼びます。


In [ ]:
for cut in [30000, 50000, 100000, 300000, 10000000]:
    price_for(cut)


あなたの部屋の値段は、3万円で切ると18,629円、外さないと27,358円と、9千円近く動きます。5節で見た補い方の違い（全体で4円、欠けた部屋で300円）よりずっと大きく、どの部屋を相場に入れるかが、値段をいちばん動かす判断でした。


### 問い：外れ幅が、どんどん小さくなる

外れ幅も、外さないときの19,157円から、3万円で切ると4,173円まで縮みます。線を厳しくするほど、式は良くなったのでしょうか。下に書いてください。


（ここに予想を書く）


良くなったとは言えません。線を変えると、テストの部屋そのものが変わるからです。3万円で切ると、当てにくい高い部屋がテストからも消え、3万円以下の部屋だけで採点しています。物差しの部屋が違う外れ幅は、比べられません。第2回で見た「採点が甘い」と同じ落とし穴です。

線は、外れ幅が小さくなるように選ぶのではなく、目標から選びます。あなたの部屋は定員4人のふつうの部屋で、1泊数十万円の一棟貸しと競うわけではありません。だから10万円の線を使い、その線を書き残します。


### 答え

あなたの部屋（定員4人・寝室2・ベッド2・バスルーム1）は、1泊およそ2万5千円（24,830円）。まだ見ていない部屋での外れ幅は、平均でおよそ8,900円。相場に入れる部屋を1泊10万円以下の貸切とした場合の値段です。


まだ言えないこと

- 外れ幅の平均だけで測ってよいのか：大外れを重く見る物差しもある
- 地区：同じ貸切でも、地区によって料金は違いそうなのに、地区（文字の列）は式に入っていない
- 補った値の揺れ：推測で補った値は1本の線に乗るので、本当の値より揃いすぎている（発展で扱う）


---


## 目標に答えられたか


- 今日の目標は「自分の部屋に、1泊いくらの値段を付ければいいか」でした。9節の答えは何円で、どれくらい外れうるでしょうか？
- 1節で、10万円を超える部屋と、貸切以外の部屋を外しました。それぞれ外した理由を、今日の目標から説明できますか？
- 3節で、レビューの点数・バスルーム数・寝室数の欠けを、それぞれどう扱いましたか？ なぜ扱いが違ったのでしょうか？
- 4節で、平均で埋めるのはなぜよくないと分かりましたか？ 外れ・図・相関の3つで説明できますか？
- 7節の`prep`は、なぜ「決める表」と「整える表」の2つを受け取るのでしょうか？


---


## 今日の要点


- 今日、持ち帰る1文：欠けた値は、1つの値で埋めない。欠けた理由を確かめ、ほかの列から推測して補う。整え方は訓練データだけで決め、関数にして、新しい部屋にも同じように当てる
- 外す行は、目標から理由を説明でき、どこで線を引いたかを書き残せるときに外す。外れ幅が小さくなるように選ばない
- 欠けた値は、理由を先に確かめる。値が存在しない欠け（レビューの点数）は埋めない。別の列から戻せる欠け（バスルーム数）は戻す
- 平均で埋めると、補った値が外れ、列どうしの関係が弱まる。小さい部屋ほど欠けるときは、平均そのものもずれる
- 答えの分かる欠けを作れば、補い方を採点できる
- 標準化しても線形回帰の予測は変わらない。係数を同じ目盛りで比べられるようになる
- Python：`def`で関数を作る。引数が入力、`return`の後ろが戻り値。既定値つきの引数で選択肢を切り替え、小さな関数を組み合わせて大きな処理を作る


---


## 次回へ


今日は、データを整えて、あなたの部屋に1泊およそ2万5千円の値段を付けました。ただ、外れ幅の平均だけで式を測ってよいのか、式の係数をどう読めばよいのかには、まだ答えていません。

次回は、場面を住宅の査定に変えます。次回の問いは、「この家は、なぜこの値段なのか」を説明する式を、どの物差しで選び、どう読むか、です。


---


## 課題（提出）


提出するもの：応用②の答えと、応用③の文章。提出フォームに入力してください。期限はありません。応用①のコードは提出しませんが、②の答えを出すために必要です。


### 応用①（変形）

4節では、ベッド数を隠して、平均で埋めた場合と、定員から推測した場合を採点しました。同じことを、寝室数でやってください。寝室数が分かっている訓練の部屋`full_bed`から、`make_missing`で`rate=0.2`の欠けを作り、平均で埋めたときと`fill_reg`で推測したときの、隠した部屋での外れの平均を表示してください。


In [ ]:
full_bed = tr[tr['寝室数'].notna()]
practice = make_missing(full_bed, '寝室数', 0.2)
miss = practice['寝室数'].isna()
truth_bed = full_bed['寝室数']
by_mean = practice['寝室数'].fillna(practice['寝室数'].mean())
by_reg = fill_reg(practice, practice, '寝室数')['寝室数']
print('平均 外れ', round(np.abs(by_mean - truth_bed)[miss].mean(), 2))
print('推測 外れ', round(np.abs(by_reg - truth_bed)[miss].mean(), 2))


<details><summary>詰まったら</summary>

まず`full_bed = tr[tr['寝室数'].notna()]`で、寝室数が分かっている部屋を取り出します。
4節の「平均で埋めて、採点する」のセルを写し、`full`を`full_bed`に、`'ベッド数'`を`'寝室数'`に変えます。本当の値も`full_bed['寝室数']`にします。推測で補うのは、5節の`fill_reg(practice, practice, '寝室数')`を使うと1行で書けます。

</details>


### 応用②（判断）

応用①で、寝室数を平均で埋めたとき、隠した部屋での外れの平均は何室ですか。小数第2位まで答えてください（例: 0.12）。


In [ ]:
print('答え:', round(np.abs(by_mean - truth_bed)[miss].mean(), 2))


<details><summary>詰まったら</summary>

`round(値, 2)`で小数第2位に丸めます。定員から推測したときの外れとも比べておきましょう。

</details>


### 応用③（解釈）

応用①の結果と4節の図をもとに、「欠けた寝室数を、平均で埋めてはいけない理由」と「代わりにどう補うか」を、民泊の新しいホストに向けて3行程度で説明してください。


（模範例）

寝室数を平均（約1.6室）で埋めると、隠した部屋での外れは平均0.67室で、定員から推測したとき（0.31室）の2倍以上になりました。

寝室数は小さい部屋ほど欠けやすいので、残った部屋の平均は大きめにずれます。その値で埋めると、ワンルームのような小さい部屋にも1.6室が入り、定員と寝室数の関係も弱まります。

欠けた寝室数は、まず欠けた理由（ワンルームを0ではなく空欄にしていないか）を確かめ、補うなら定員のような関係の強い列から推測します。


---


## 発展（任意）


### 推測に揺らぎを足して補う：多重代入の考え方

4節で、定員から推測して補うと、相関が本当の0.848より強めに出ました（`rate=0.2`で0.863）。推測した値は、どれも1本の線の上にぴったり乗るからです。本当のベッド数は、同じ定員でも部屋ごとに散らばっています。推測で補った部屋だけ、散らばりが消えているのです。

そこで、推測した値に、本当の値と推測の外れと同じくらいの揺らぎを足して補います。

$$
\tilde{x}_i = \hat{x}_i + \varepsilon_i, \qquad \varepsilon_i \sim N(0,\, s^2)
$$

$\hat{x}_i$は推測、$\varepsilon_i$は平均0・標準偏差$s$の正規分布に従う乱数です。$s$は、値の分かる部屋での推測の外れ$x_k - \hat{x}_k$の標準偏差にします。

- 役割：推測の外れの散らばりを測り、その大きさの乱数を推測に足して補う
- 入力：4節の`full`と`make_missing`・`fill_reg`
- 出力：揺らぎを足したときの相関と標準偏差

`rng.normal(0, s, 個数)`は、平均0・標準偏差`s`の正規分布に従う乱数を、指定した個数だけ出します。


In [ ]:
truth = full['ベッド数']
practice = make_missing(full, 'ベッド数', 0.2)
miss = practice['ベッド数'].isna()
ok = practice[practice['ベッド数'].notna()]
m = LinearRegression().fit(ok[['定員']], ok['ベッド数'])
# 推測の外れの散らばり（標準偏差）
s = (ok['ベッド数'] - m.predict(ok[['定員']])).std()
print('推測の外れの標準偏差:', round(s, 2))

by_reg = fill_reg(practice, practice, 'ベッド数')['ベッド数']
rng = np.random.default_rng(1)
by_noise = by_reg.copy()
by_noise[miss] = by_reg[miss] + rng.normal(0, s, miss.sum())
for name, val in [('本当の値', truth), ('推測', by_reg),
                  ('推測＋揺らぎ', by_noise)]:
    print(name, ' 相関', round(full['定員'].corr(val), 3),
          ' 標準偏差', round(val.std(), 2))


In [ ]:
plt.scatter(full['定員'][miss], by_noise[miss], s=4, alpha=0.3)
plt.xlabel('定員')
plt.ylabel('揺らぎを足して補ったベッド数')
plt.show()


読み方　揺らぎを足すと、相関も標準偏差も本当の値に近づきます。図の点も、1本の線ではなく、本当の値のように散らばります。ただし、0より小さいベッド数も出てしまいます。正規分布の揺らぎをそのまま足しただけの、単純なやり方の弱点です。

ただし、足した揺らぎは乱数なので、種を変えれば補った値も変わります。そこで、種を変えて補ったデータを何通りも作り、それぞれで同じ分析をして、結果をまとめます。補った値の不確かさを、結果の幅として残せるのです。この方法を多重代入（multiple imputation）と呼びます。


In [ ]:
for seed in range(5):
    rng = np.random.default_rng(seed)
    by_noise = by_reg.copy()
    by_noise[miss] = by_reg[miss] + rng.normal(0, s, miss.sum())
    print(seed, by_noise[miss].head(3).round(1).values,
          ' 相関', round(full['定員'].corr(by_noise), 3))


隠した部屋の最初の3件に入る値は、種ごとに違います。それでも相関は、どの種でも0.833で、本当の0.848に近いままです。試すなら、`rate`を0.4に上げて、推測だけで補ったときと、揺らぎを足したときの相関を比べてみましょう。
